# Powerful Design of Small Vision Transformer on CIFAR10 — reproduction on Kaggle (2× T4)

This notebook reproduces **Table 2** (augmentation, patch-token init, MLA low-rank attention, DDP / Lion) and **Table 3** (multi-CLS tokens) of the paper. It:

1. clones the fixed code from GitHub,
2. downloads CIFAR-10 from the Hugging Face CDN (≈145 MB, a few seconds),
3. benchmarks the T4s and **sizes the number of epochs so every experiment finishes inside the time budget** (paper: 100 epochs on an RTX 3090 — not affordable for 16 runs on 2× T4 in 4 h),
4. trains all 16 runs (two single-GPU runs in parallel, one per T4, then the 3 DDP runs on both T4s),
5. compares the results with the paper.

Only the metrics the paper reports are computed: **test top-1** on the full CIFAR-10 test set (every `EVAL_FREQ` epochs + the last epoch -> `best_acc` / `final_acc`) and **runtime**, plus the mean train loss per epoch (GPU-side) to spot diverging runs. No train accuracy, no validation loss and no separate re-evaluation pass, so nearly all GPU time goes to training.

**Before running:** *Settings → Accelerator → GPU T4 x2* and *Settings → Internet → On*. Then **Save Version → Save & Run All (Commit)** so it keeps running in the background (≈ 3 h 50 min).

In [ ]:
# ---------------- configuration ----------------
REPO_URL     = "https://github.com/Param45/PoorViTs.git"
BRANCH       = "main"
BUDGET_HOURS = 3.6     # benchmark + all training runs; setup adds ~3 min on top -> < 4 h total
EXPERIMENTS  = "all"   # or e.g. "baseline,no_aa,mla_q,dim96_cls2"   (python run_experiments.py --list)
EPOCHS       = 0       # 0 = largest epoch count that fits the budget (capped at the paper's 100)
EVAL_FREQ    = 5       # test-set top-1 every N epochs (+ the last epoch)

WORK    = "/kaggle/working"
REPO    = f"{WORK}/PoorViTs"
DATA    = f"{WORK}/data"
RESULTS = f"{WORK}/results"
import time; T0 = time.time()

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch; print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 1, "No GPU: Settings -> Accelerator -> GPU T4 x2"

In [ ]:
# get the code
!rm -rf {REPO} && git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!pip install -q gin-config lion-pytorch einops colorama
%cd {REPO}
!git log -1 --oneline

In [ ]:
# download CIFAR-10 (Hugging Face CDN parquet -> uint8 .npz; falls back to the original Toronto server)
t = time.time()
!python utils/cifar_data.py --datapath {DATA} --dataset CIFAR10
print(f"dataset ready in {time.time() - t:.0f}s")

## Train all experiments

Progress is printed every 10 minutes; each run's full log is in `results/<experiment>/train.log`.
If the notebook gets interrupted, re-running this cell skips the runs that already finished.

In [ ]:
!python run_experiments.py --datapath {DATA} --output_dir {RESULTS} --budget_hours {BUDGET_HOURS} --experiments {EXPERIMENTS} --epochs {EPOCHS} --eval_freq {EVAL_FREQ}

## Results vs. the paper

In [ ]:
import pandas as pd
from IPython.display import Markdown, Image, display
# test top-1 is measured on the full CIFAR-10 test set during training:
# best_acc = best evaluated epoch, final_acc = last epoch (from each run's results.json)
summary = pd.read_csv(f"{RESULTS}/summary.csv")
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
display(summary)
display(Markdown(open(f"{RESULTS}/summary.md").read().split("\n", 2)[2]))
display(Image(f"{RESULTS}/curves.png"))

In [ ]:
# pack logs + results (checkpoints of the baseline only, to keep it small) for download from the Output tab
!cd {WORK} && zip -qr results_logs.zip results -x "results/*/checkpoint.pth" -x "results/*/best.pth" -x "results/_bench/*" && zip -qj results_logs.zip results/baseline/best.pth 2>/dev/null; ls -lh {WORK}/results_logs.zip
print(f"total notebook time: {(time.time() - T0) / 3600:.2f} h")